# Machine Overview

Establish a reproducible, read-only baseline of the machine before running performance experiments.

## Objectives

- Identify the CPU, memory, GPU, NUMA, storage, and PCIe topology.
- Record command availability and failures explicitly.
- Separate expected DGX Spark properties from measured observations.

## Background

The current DGX Spark baseline is expected to have an `aarch64` architecture, 20 CPU cores with heterogeneous Cortex-X925 and Cortex-A725 cores, one NUMA node, approximately 128 GB of unified LPDDR5X memory, an NVIDIA GB10 GPU, ConnectX-7 networking, and NVMe storage. These are properties to verify below, not recorded experimental results.

## Prediction

TODO: Before executing the command cells, state which expected properties each command should confirm and note any uncertainty.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

The following commands are read-only and do not require root privileges. A missing utility or nonzero exit is part of the result and does not stop the remaining checks.

In [2]:
from common import CommandResult, run_command

commands = (
    ("lscpu",),
    ("free", "-h"),
    ("numactl", "--hardware"),
    ("nvidia-smi",),
    ("uname", "-a"),
    ("lsblk",),
    ("lspci",),
)


def print_command_result(result: CommandResult) -> None:
    print(f"\n$ {' '.join(result.command)}")
    print(f"exit status: {result.returncode}")
    print(f"executable missing: {result.executable_missing}")
    print(f"timed out: {result.timed_out}")
    print("stdout:")
    print(result.stdout or "<empty>")
    print("stderr:")
    print(result.stderr or "<empty>")
    if result.error:
        print(f"error: {result.error}")


machine_results = [run_command(command, timeout=30.0) for command in commands]
for machine_result in machine_results:
    print_command_result(machine_result)


$ lscpu
exit status: 0
executable missing: False
timed out: False
stdout:
Architecture:                            aarch64
CPU op-mode(s):                          64-bit
Byte Order:                              Little Endian
CPU(s):                                  20
On-line CPU(s) list:                     0-19
Vendor ID:                               ARM
Model name:                              Cortex-X925
Model:                                   1
Thread(s) per core:                      1
Core(s) per socket:                      10
Socket(s):                               1
Stepping:                                r0p1
Frequency boost:                         disabled
CPU(s) scaling MHz:                      100%
CPU max MHz:                             3900.0000
CPU min MHz:                             1378.0000
BogoMIPS:                                2000.00
Flags:                                   fp asimd evtstrm aes pmull sha1 sha2 crc32 atomics fphp asimdhp cpuid asimdrdm

## Observations

TODO: After running the cells, record the observed topology and any missing or failing commands. Do not copy the expected baseline here without measurement.

## Explanation

TODO: Reconcile each observation with the prediction and explain discrepancies using the command output.

## Connection to LLMs

CPU topology, unified memory capacity, GPU capabilities, storage, and network interfaces constrain model loading, preprocessing, inference throughput, and multi-node communication.

## Further Exploration

TODO: Identify one topology detail to verify with a more targeted, non-privileged experiment.